### Lab 8.1 RNN Implementation

This week we will work up to creating an RNN text generator.  In today's lab you will implement a basic RNN.

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

1. Here I have predefined the RNN weights and input sequence for you.  The configuration is as follows:

* N = 5 is the sequence length
* C = 3 is number of input channels
* H = 10 is size of hidden state vector
* O = 1 is number of output channels

Compute the result of running the RNN on the input sequence, using ReLU activation on the hidden state.  The result should have shape [N,O] = [5,1] and contain these values:

```
0.7056, 1.0312, 4.8223, -7.6193, -11.9787
```

In [ ]:
torch.manual_seed(42)

x = torch.randn(5,3) # input sequence [N,C]

U = torch.randn(10,3) # U matrix [H,C]
W = torch.randn(10,10) # W matrix [H,H]
V = torch.randn(1,10) # V matrix [O,H]

h = torch.zeros(10) # initial hidden state [H]

In [5]:
ys = []
for t in range(x.shape[0]):
    h = torch.relu(U @ x[t] + W @ h)
    y = V @ h
    ys.append(y)

y = torch.stack(ys) 
print(y)

tensor([[  0.7056],
        [  1.0312],
        [  4.8223],
        [ -7.6193],
        [-11.9787]])


2. Now adapt your RNN implementation into a Torch module.   Run the code to verify that the output shape is [1,5,1].

In [8]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
      super().__init__()

      self.hidden_size = hidden_size
      self.U = nn.Linear(input_size,hidden_size,bias=False)
      self.W = nn.Linear(hidden_size,hidden_size,bias=False)
      self.V = nn.Linear(hidden_size,output_size,bias=False)

    def forward(self,x):
      # x is a batch of sequences of shape [B,N,C]
      # output shape should be [B,N,O]
      B,N,C = x.shape
      h = torch.zeros(B, self.hidden_size)

      ys = []
      for t in range(N):
          h = F.relu(self.U(x[:, t, :]) + self.W(h))
          ys.append(self.V(h))

      return torch.stack(ys, dim=1) 


model = RNN(3,10,1)

model(x.unsqueeze(0))

tensor([[[0.0417],
         [0.2481],
         [0.1074],
         [0.0993],
         [0.6976]]], grad_fn=<StackBackward0>)